# MPE MIDI File Player - 53-TET Dataset

This notebook allows you to listen to MIDI files from the 53-TET MPE dataset.

## Features:
- List all available MIDI files
- Play a random file from the dataset
- Play a specific file by index or search by name
- Control playback speed

The files use MIDI Polyphonic Expression (MPE) to accurately represent microtonal pitches in the 53-tone equal temperament system.


In [ ]:
# Import necessary modules
import os
import random
from pathlib import Path
import sys
import importlib

# Get the current working directory and add to path
current_dir = Path.cwd()
if current_dir.name == "src":
    # Already in the src directory
    sys.path.insert(0, str(current_dir))
else:
    # Assume we're in the project rootimport importlib
    sys.path.insert(0, str(current_dir / "src"))

# Import the play_mpe functions
# reload play_me
import play_mpe as pm
importlib.reload(pm)

# Set the dataset path (relative to project root)
if current_dir.name == "src":
    DATASET_PATH = Path("../dataset/midi_files/53_tet_mpe")
else:
    DATASET_PATH = Path("dataset/midi_files/53_tet_mpe")
    
print(f"Dataset path: {DATASET_PATH.resolve()}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

path ="/dataset/midi_files/53_tet_mpe/type_0_major"

In [ ]:
# Get all MIDI files from the dataset
midi_files = sorted([f for f in DATASET_PATH.glob("**/*.mid")])
print(f"Found {len(midi_files)} MIDI files in the dataset")
print(f"\nFirst 10 files:")
for i, f in enumerate(midi_files[:10], 1):
    print(f"  {i}. {f.name}")


In [ ]:
# Dataset statistics
from collections import Counter

# Extract information from filenames
types = []
keys = []
voicings = []

for f in midi_files:
    parts = f.stem.split("_")
    if len(parts) >= 4:
        # Extract key (e.g., C, Db, etc.)
        if len(parts) > 2:
            keys.append(parts[2])
        # Extract type
        if "type" in f.stem:
            type_parts = [p for p in parts if "type" in p]
            if type_parts:
                types.append(type_parts[0])
        # Extract voicing (last part)
        voicings.append(parts[-1])

print("=== Dataset Statistics ===\n")
print(f"Total files: {len(midi_files)}")
print(f"\nTop 10 keys:")
for key, count in Counter(keys).most_common(10):
    print(f"  {key}: {count} files")

print(f"\nTop 10 voicing types:")
for voicing, count in Counter(voicings).most_common(10):
    print(f"  {voicing}: {count} files")


In [ ]:
# MIDI VISUALIZATION + AUDIO

import midi_viz as mv 
importlib.reload(mv)
importlib.reload(pm)
from IPython.display import Audio, display

# Pick a file (random or by index)
# test_file = random.choice(midi_files)  # or midi_files[0] for first file
target = "11703_Everything Must Change_Eb_minor_type_1_minor.mid"
#find target
test_file = None
for f in midi_files:
    if f.name == target:
        test_file = f
        break

if test_file is None:
    raise FileNotFoundError(f"File {target} not found in midi_files.")

print(f"🎵 Playing: {test_file.name}\n")



In [ ]:
# Show piano roll visualization (first 30 seconds)
print("📊 Creating visualization...")   
fig = mv.visualize_midi(test_file, speed=1.5, max_duration=120)
if fig:
    fig.show()

# Play audio
print("\n🎧 Rendering audio...")
audio_data, sr = pm.render_mpe_to_audio_data(test_file, speed=1.5, waveform='square', reverb=25)
if audio_data is not None:
    display(Audio(audio_data, rate=sr))

In [ ]:
# ── Diagnose stuck notes in the current test_file ──
import mido

mid = mido.MidiFile(str(test_file))
tpb = mid.ticks_per_beat

active = {}
stuck = {}
for track in mid.tracks:
    abs_time = 0
    for msg in track:
        abs_time += msg.time
        if msg.type == 'note_on' and msg.velocity > 0:
            k = (msg.channel, msg.note)
            if k in active:
                stuck[k] = active[k]  # channel reuse → stuck
            active[k] = abs_time
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            active.pop((msg.channel, msg.note), None)

# Notes still active at EOF = also stuck
for k, onset in active.items():
    stuck[k] = onset

if stuck:
    print(f'⚠️  {len(stuck)} STUCK NOTES in {test_file.name}:\n')
    for (ch, note), onset_tick in sorted(stuck.items(), key=lambda x: x[1]):
        print(f'  ch={ch:>2}  midi={note:>3}  from beat {onset_tick/tpb:>7.1f}  (never gets note_off)')
    print(f'\nThese ring forever → long tail you hear.')
    print(f'Root cause: generate_53tet_dataset.py reuses channels after 15 notes.')
else:
    print(f'✅ No stuck notes in {test_file.name}')

In [ ]:
# ── Replay with REVERTED renderer ──
importlib.reload(pm)
importlib.reload(mv)

print('📊 Piano roll...')
fig2 = mv.visualize_midi(test_file, speed=1.5, max_duration=120)
if fig2:
    fig2.show()

print('\n🎧 Audio...')
audio_fixed, sr2 = pm.render_mpe_to_audio_data(test_file, speed=1.5, waveform='square', reverb=25)
if audio_fixed is not None:
    display(Audio(audio_fixed, rate=sr2))

In [ ]:
# ── Scan dataset sample for stuck notes ──
import random as _rng
_rng.seed(42)
sample = _rng.sample(midi_files, min(3000, len(midi_files)))

stuck_count = 0
for sf in sample:
    m = mido.MidiFile(str(sf))
    act = {}
    for t in m.tracks:
        at = 0
        for msg in t:
            at += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                act[(msg.channel, msg.note)] = at
            elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                act.pop((msg.channel, msg.note), None)
    if act:
        stuck_count += 1

pct = stuck_count / len(sample) * 100
est = int(pct / 100 * len(midi_files))
print(f'Files with stuck notes: {stuck_count}/{len(sample)} ({pct:.1f}%)')
print(f'Estimated across full dataset: ~{est:,} / {len(midi_files):,}')
print(f'\nplay_mpe.py and midi_viz.py are now fixed to handle these correctly.')

In [ ]:
# ── DIAGNOSE: chord durations vs gap to next chord ──
import mido
from pathlib import Path

# Load tokenizer's parse_mpe_midi
import sys
spec_path = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
if str(spec_path) not in sys.path:
    sys.path.insert(0, str(spec_path))
from tokenizer import parse_mpe_midi

# Parse test file
chords = parse_mpe_midi(test_file)
print(f'File: {test_file.name}')
print(f'Total chords: {len(chords)}\n')

# Find chords whose duration exceeds the gap to next chord
problems = []
for i in range(len(chords) - 1):
    gap = chords[i+1]['onset_beats'] - chords[i]['onset_beats']
    if chords[i]['duration_beats'] > gap + 0.01:
        problems.append((i, chords[i]['duration_beats'], gap))

print(f'Chords with duration > gap to next: {len(problems)} / {len(chords)}\n')
print(f'{"idx":>4}  {"onset":>8}  {"dur":>8}  {"gap":>8}  {"excess":>8}  notes')
print(f'{"---":>4}  {"---":>8}  {"---":>8}  {"---":>8}  {"---":>8}  -----')
for idx, dur, gap in problems[:25]:
    c = chords[idx]
    print(f'{idx:>4}  {c["onset_beats"]:>8.1f}  {dur:>8.1f}  {gap:>8.1f}  {dur-gap:>8.1f}  {len(c["notes"])}')
if len(problems) > 25:
    print(f'  ... and {len(problems)-25} more')

max_dur = max(c['duration_beats'] for c in chords)
print(f'\nMax chord duration: {max_dur:.1f} beats')

In [ ]:
# ── STEP 2: Fix ONLY the broken notes in the test file ──
# For each clobbered/orphaned note: insert a note_off at next chord onset.
# Nothing else is touched.

import mido, tempfile
from pathlib import Path

def fix_broken_notes(midi_path, out_path=None):
    """Insert missing note_off for clobbered and orphaned notes."""
    mid = mido.MidiFile(str(midi_path))
    tpb = mid.ticks_per_beat
    TOL = max(1, tpb // 48)

    # Collect chord onsets
    on_ticks = set()
    for track in mid.tracks:
        at = 0
        for msg in track:
            at += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                on_ticks.add(at)
    chord_onsets = sorted(on_ticks)

    def next_chord_after(tick):
        for c in chord_onsets:
            if c > tick + TOL:
                return c
        return None

    stats = {'clobbered': 0, 'orphans': 0}

    for track in mid.tracks:
        active = {}     # (ch, note) -> onset_tick
        inserts = []    # (abs_tick, note_off_msg)

        at = 0
        for msg in track:
            at += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                key = (msg.channel, msg.note)
                if key in active:
                    # Clobbered: close old note at next chord after its onset
                    nxt = next_chord_after(active[key])
                    off_tick = min(at, nxt) if nxt else at
                    inserts.append((off_tick, mido.Message(
                        'note_off', channel=key[0], note=key[1], velocity=0)))
                    stats['clobbered'] += 1
                active[key] = at
            elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                active.pop((msg.channel, msg.note), None)

        # Orphaned: still active at EOF
        for (ch, note), onset in active.items():
            nxt = next_chord_after(onset)
            inserts.append((nxt if nxt else at, mido.Message(
                'note_off', channel=ch, note=note, velocity=0)))
            stats['orphans'] += 1

        # Insert the missing note_offs into the track
        if inserts:
            abs_msgs = []
            t = 0
            for msg in track:
                t += msg.time
                abs_msgs.append((t, msg))
            for tick, msg in inserts:
                abs_msgs.append((tick, msg))
            abs_msgs.sort(key=lambda x: x[0])
            track.clear()
            prev = 0
            for t, msg in abs_msgs:
                msg.time = t - prev
                track.append(msg)
                prev = t

    if out_path:
        mid.save(str(out_path))
    return mid, stats

# ── Apply to test file (temp copy) ──
tmp_dir = Path(tempfile.mkdtemp())
fixed_path = tmp_dir / test_file.name

_, st = fix_broken_notes(test_file, fixed_path)
print(f'Fixed: {test_file.name}')
print(f'  Clobbered fixed: {st["clobbered"]}')
print(f'  Orphans fixed:   {st["orphans"]}')

# Verify
mid2 = mido.MidiFile(str(fixed_path))
act2 = {}; c2 = 0
for track in mid2.tracks:
    at = 0
    for msg in track:
        at += msg.time
        if msg.type == 'note_on' and msg.velocity > 0:
            if (msg.channel, msg.note) in act2: c2 += 1
            act2[(msg.channel, msg.note)] = at
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            act2.pop((msg.channel, msg.note), None)
print(f'\nVerify: clobbered={c2} orphaned={len(act2)}')
if c2 == 0 and len(act2) == 0:
    print('✅ CLEAN')

# Listen
importlib.reload(pm)
print('\n🎧 Original:')
a_o, sr_o = pm.render_mpe_to_audio_data(test_file, speed=1.5, waveform='square', reverb=25)
if a_o is not None: display(Audio(a_o, rate=sr_o))

print('🎧 Fixed:')
a_f, sr_f = pm.render_mpe_to_audio_data(fixed_path, speed=1.5, waveform='square', reverb=25)
if a_f is not None: display(Audio(a_f, rate=sr_f))


In [ ]:
# ── STEP 3: Test on a second file ──
import random
random.seed(77)
second_file = random.choice([f for f in midi_files if f.name != target])
print(f'Second: {second_file.name}\n')

fixed2 = tmp_dir / second_file.name
_, st2 = fix_broken_notes(second_file, fixed2)
print(f'Clobbered: {st2["clobbered"]}  Orphans: {st2["orphans"]}')

# Verify
m3 = mido.MidiFile(str(fixed2))
a3 = {}; c3 = 0
for track in m3.tracks:
    at = 0
    for msg in track:
        at += msg.time
        if msg.type == 'note_on' and msg.velocity > 0:
            if (msg.channel, msg.note) in a3: c3 += 1
            a3[(msg.channel, msg.note)] = at
        elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
            a3.pop((msg.channel, msg.note), None)
print(f'Verify: clobbered={c3} orphaned={len(a3)}')
if c3 == 0 and len(a3) == 0: print('✅ CLEAN')

print('\n🎧 Original:')
ao, so = pm.render_mpe_to_audio_data(second_file, speed=1.5, waveform='square', reverb=25)
if ao is not None: display(Audio(ao, rate=so))
print('🎧 Fixed:')
af, sf2 = pm.render_mpe_to_audio_data(fixed2, speed=1.5, waveform='square', reverb=25)
if af is not None: display(Audio(af, rate=sf2))


In [ ]:
# ── STEP 4: Fix entire dataset (in-place) ──
# ONLY after confirming steps 2 & 3.
from tqdm import tqdm

all_midi = sorted(DATASET_PATH.glob('**/*.mid'))
print(f'Files: {len(all_midi)}')

fixed = 0; clean = 0; errs = []
for f in tqdm(all_midi, desc='Fixing'):
    try:
        _, s = fix_broken_notes(f, f)
        if s['clobbered'] + s['orphans'] > 0: fixed += 1
        else: clean += 1
    except Exception as e:
        errs.append((f.name, str(e)))

print(f'\nFixed: {fixed}  Clean: {clean}  Errors: {len(errs)}')
for n, e in errs[:5]: print(f'  {n}: {e}')
